# From a molecule to an embedded active space

Notebook 1 built the embedding and notebook 3 chose the active space. This one is the short path
straight through both, on a molecule big enough that the fragment is a genuine minority of the
system, with everything configurable in one cell.

```
global DFT  ->  localise  ->  pick the fragment  ->  embedded HF  ->  choose a CAS  ->  CASCI  ->  qubits
     (nbed.act_env_space)         (nbed.emb_scf)        (nbed.cas_space)        (nbed.hamiltonian)
```

Acetaldehyde, CH$_3$CHO, with the **carbonyl group as the fragment**: 50 basis functions and 24
electrons, of which the fragment keeps 8. The C=O $\pi/\pi^*$ pair is the interesting part and the
methyl group is along for the ride, which is exactly the situation embedding is for.

## Two independent things get frozen, at two different levels

This is the point worth keeping straight, because notebook 1 and notebook 3 each freeze something
different and here both happen at once:

| what | frozen at | chosen by | why |
|---|---|---|---|
| environment orbitals | **DFT**, by the projector | `N_OCC_FRAG` | the environment is not interesting and DFT is cheap |
| fragment core orbitals | **HF**, as CASCI `ncore` | `N_CAS_OCC` | inside the fragment, not every occupied orbital is correlated |

Neither is deleted. The environment keeps contributing $v_\text{emb}$ to the fragment's one-electron
Hamiltonian, and the frozen fragment core keeps contributing its Coulomb and exchange potential to
the CAS, with its energy folded into `ecore`. Setting `N_CAS_OCC = None` would make every fragment
occupied orbital active and leave `ncore = 0`.

## The three projectors

All three hide the environment's occupied orbitals from the fragment's aufbau filling, differently:

| `PROJ_TYPE` | environment placed at | exact? | environment columns land |
|---|---|---|---|
| `"mu"` | $+\mu$, a number you choose | no, error $\mathcal{O}(1/\mu)$ | contiguous, at the end |
| `"huz"` | $-\varepsilon_\text{env}$, set by the system | yes, to machine precision | scattered among the active virtuals |
| `"huz+shift"` | $-\varepsilon_\text{env} + \lambda$ | yes | contiguous, at the end |

Change `PROJ_TYPE` in the next cell and re-run: `"huz"` and `"huz+shift"` land within $10^{-10}$ Ha
of each other, while `"mu"` sits $1.2\times10^{-7}$ Ha away with `MU_VAL = 1e6`, which is the $1/\mu$
error and not a bug.

In [ ]:
import nbed.backend
nbed.backend.set_backend(use_gpu=False)
if nbed.backend.USING_GPU:
    print("Running on GPU")
    import cupy as np
    from gpu4pyscf import dft, scf
    from pyscf import gto, lo, mcscf
else:
    print("Running on CPU")
    import numpy as np
    from pyscf import cc, ci, dft, gto, lo, mcscf, scf

np.set_printoptions(linewidth=110, suppress=True)
import numpy

Running on CPU


In [2]:
# ============================== the system ===============================
GEOMETRY = [
    ("C", ( 0.000000,  0.000000,  0.000000)),   # carbonyl carbon
    ("O", ( 0.000000,  1.216000,  0.000000)),
    ("C", ( 1.242919, -0.841519,  0.000000)),   # methyl carbon
    ("H", (-0.952962, -0.561337,  0.000000)),   # aldehyde hydrogen
    ("H", ( 0.967106, -1.898113,  0.000000)),
    ("H", ( 1.832919, -0.619313, -0.891621)),
    ("H", ( 1.832919, -0.619313,  0.891621)),
]
BASIS = "6-31G*"
XC = "b3lyp"

# ============================= the fragment =============================
FRAGMENT = [0, 1]        # carbonyl C and O, 0-based atom indices
N_OCC_FRAG = 4           # fragment occupied orbitals -> the fragment gets 8 electrons
N_VIR_FRAG = 4           # only defines the embedding block; the CAS is chosen later
MAX_SPREAD = 3.0         # reject orbitals more diffuse than this, in Bohr

# ============================ the projector =============================
PROJ_TYPE = "huz+shift"  # "mu" | "huz" | "huz+shift"
# PROJ_TYPE = "mu"
# PROJ_TYPE = "huz"

MU_VAL = 1e6
HUZ_SHIFT = 1e6

# =========================== the active space ===========================
N_CAS_OCC = 2            # active occupied; the other N_OCC_FRAG - N_CAS_OCC become frozen core
N_CAS_VIR = 3            # active virtuals, chosen by MP2 natural occupation
N_POOL = 5               # virtuals MP2 screens over (section 4 checks this is sensible)

# ========================================================================
PROJ_KWARGS = {
    "mu":        dict(proj_type="mu"),
    "huz":       dict(proj_type="huz", huz_level_shift=0.0),
    "huz+shift": dict(proj_type="huz", huz_level_shift=HUZ_SHIFT),
}[PROJ_TYPE]
N_ACT_OCC = N_OCC_FRAG if N_CAS_OCC is None else N_CAS_OCC   # None means "keep them all"
N_FROZEN_CORE = N_OCC_FRAG - N_ACT_OCC

print(f"projector {PROJ_TYPE!r};  CAS({2 * N_ACT_OCC}e, {N_ACT_OCC + N_CAS_VIR}o) "
      f"= {2 * (N_ACT_OCC + N_CAS_VIR)} qubits;  {N_FROZEN_CORE} frozen fragment core orbitals")

projector 'huz+shift';  CAS(4e, 5o) = 10 qubits;  2 frozen fragment core orbitals


In [ ]:
from nbed.act_env_space import lowdin_populations, orbital_spread
from nbed.cas_space import (diagnose_cas, environment_overlap, report_cas_orbitals,
                            run_casci, select_cas_by_mp2_no, valence_virtual_rotation)
from nbed.emb_scf import EmbedSCF

mol = gto.M(atom=GEOMETRY, basis=BASIS, charge=0, spin=0, max_memory=10_000, verbose=0)
print(f"nao = {mol.nao}   nelec = {mol.nelec}   "
      f"fragment atoms {[mol.atom_symbol(i) + str(i) for i in FRAGMENT]}")

nao = 50   nelec = (12, 12)   fragment atoms ['C0', 'O1']


## 1. Global DFT, then localise

The cheap calculation everything else is built from. Localisation is a unitary rotation *within*
each occupation block, so it leaves the density and therefore the global DFT energy untouched, but
it turns delocalised canonical orbitals into recognisable bonds and lone pairs. Without it "the C=O
group" is not any single orbital and the partition would be fuzzy.

In [ ]:
glob = dft.RKS(mol, xc=XC)
glob.verbose = 0
glob.kernel()
ovlp = glob.get_ovlp()
assert glob.converged, "global DFT did not converge"
print(f"global {XC.upper()} = {glob.e_tot:.10f}")


if nbed.backend.USING_GPU:
    mo_coeff = numpy.asarray(glob.mo_coeff.get())
    mo_occ = numpy.asarray(glob.mo_occ.get())
else:
    mo_coeff = glob.mo_coeff.copy()
    mo_occ   = glob.mo_occ.copy()

C_loc = mo_coeff.copy()
for mask in (mo_occ > 1, mo_occ == 0):
    if mask.sum() > 1:
        C_loc[:, mask] = lo.PipekMezey(mol, mo_coeff[:, mask]).kernel()

if nbed.backend.USING_GPU:
    C_loc = nbed.backend.xp.asarray(glob.mo_coeff)

assert np.allclose(glob.make_rdm1(mo_coeff=C_loc, mo_occ=glob.mo_occ),
                   glob.make_rdm1(), atol=1e-9), "localisation changed the density"
print("localised within occupation blocks; density unchanged")

global B3LYP = -153.8248741391
localised within occupation blocks; density unchanged


## 2. Which orbitals are the fragment

`lowdin_populations` scores each MO by the fraction of it sitting on the target atoms, in an
$S^{1/2}$-orthogonalised basis so the score is a genuine fraction in $[0, 1]$. Occupied and virtual
orbitals are ranked separately.

**Only the occupied partition matters for the embedding**: it fixes the fragment electron count and
the environment occupied orbitals build the projector. The printed populations are the thing to
sanity-check — a fragment orbital should score near 1, and an orbital shared with the environment
lands near 0.5 and should be left out.

In [5]:
_, pop = lowdin_populations(mol, C_loc, FRAGMENT, drop_core_1s=True)
eligible = orbital_spread(mol, C_loc) <= MAX_SPREAD
occ_pool = np.where((glob.mo_occ > 0) & eligible)[0]
vir_pool = np.where((glob.mo_occ == 0) & eligible)[0]

pick_occ = np.sort(occ_pool[np.argsort(-pop[occ_pool])[:N_OCC_FRAG]])
pick_vir = np.sort(vir_pool[np.argsort(-pop[vir_pool])[:N_VIR_FRAG]])
active = np.concatenate([pick_occ, pick_vir])
environment = np.setdiff1d(np.arange(mol.nao), active)

print(f"occupied populations on {FRAGMENT}, all {int((glob.mo_occ > 0).sum())} of them:")
print(f"  {np.round(pop[glob.mo_occ > 0], 3)}")
print(f"\nfragment occupied  {pick_occ}   populations {np.round(pop[pick_occ], 3)}")
print(f"left to environment: populations "
      f"{np.round(np.sort(pop[np.setdiff1d(np.where(glob.mo_occ > 0)[0], pick_occ)])[::-1], 3)}")
print(f"\n-> fragment keeps {2 * N_OCC_FRAG} of {mol.nelectron} electrons")

occupied populations on [0, 1], all 12 of them:
  [0.021 0.024 0.002 0.979 0.489 0.547 0.019 0.033 0.986 0.017 0.993 0.968]

fragment occupied  [ 3  8 10 11]   populations [0.979 0.986 0.993 0.968]
left to environment: populations [0.547 0.489 0.033 0.024 0.021 0.019 0.017 0.002]

-> fragment keeps 8 of 24 electrons


## 3. Embed, and check the embedding is exact

Two runs of the same machinery. **DFT-in-DFT** puts DFT back on the fragment, so it must reproduce
the global DFT energy exactly — that is the test that the partition and projector are sound, and it
is the only number here with a known right answer. **HF-in-DFT** then swaps in Hartree-Fock, which
is a genuine WF-in-DFT energy and is *not* supposed to match anything.

`build_emb_hf` also reports where the environment ended up, in `env_cols`. Those columns are what
everything downstream must avoid.

In [6]:
emb = EmbedSCF(glob, active, environment, C_loc, glob.mo_occ, ovlp, 10_000, mu_val=MU_VAL)

e_dft_in_dft, *_ = emb.build_emb_dft(XC, **PROJ_KWARGS)
e_hf_in_dft, mf, _, env_cols, env_plus_corr = emb.build_emb_hf(**PROJ_KWARGS)

err = e_dft_in_dft - glob.e_tot
print(f"\nDFT-in-DFT    = {e_dft_in_dft:.10f}")
print(f"global DFT    = {glob.e_tot:.10f}")
print(f"  difference  = {err:+.2e}   <- exactness test for PROJ_TYPE = {PROJ_TYPE!r}")
print(f"\nHF-in-DFT     = {e_hf_in_dft:.10f}   (subsystem nelec {mf.mol.nelec})")
print(f"environment landed in columns {env_cols}")

tol = 1e-5 if PROJ_TYPE == "mu" else 1e-8
assert abs(err) < tol, f"embedding is not exact: {err:.2e}"
if PROJ_TYPE == "mu":
    print(f"\n(the {abs(err):.1e} residual is the mu-shift's O(1/mu) error, not a bug; "
          "huz removes it)")


DFT-in-DFT    = -153.8248741390
global DFT    = -153.8248741391
  difference  = +3.35e-11   <- exactness test for PROJ_TYPE = 'huz+shift'

HF-in-DFT     = -153.4345868701   (subsystem nelec (4, 4))
environment landed in columns [42 43 44 45 46 47 48 49]


## 4. Is the screening pool meaningful?

Before choosing a CAS, one check that is easy to skip and bites quietly. `cas_space` ranks the
environment-free virtuals by how much of the fragment's **minimal valence space** each one covers,
then screens the best `N_POOL` of them with MP2. That ranking is only informative while the weights
are non-zero: a small fragment has a small valence space, and a pool reaching past it is padded with
orbitals that have *no* valence character to be ordered by. Which of them gets in is then decided by
round-off in a degenerate eigenvector, and the final energy stops being reproducible at the
$10^{-4}$ Ha level — including between `"huz"` and `"huz+shift"`, which are otherwise identical.

The C=O fragment has 4 occupied orbitals out of a minimal C+O valence space of 8, so only a handful
of valence virtuals exist. `select_cas_by_mp2_no` warns if `N_POOL` overruns them; this cell shows
the spectrum so the cut can be placed by eye.

In [7]:
vir_safe = np.setdiff1d(np.where(mf.mo_occ == 0)[0], env_cols)
_, weights, _ = valence_virtual_rotation(mf.mol, mf.mo_coeff[:, vir_safe], FRAGMENT)
n_orderable = int((weights > 1e-6).sum())     # past this the ranking is round-off
n_substantial = int((weights > 0.01).sum())   # genuine valence character

print(f"{len(vir_safe)} environment-free virtuals; fragment valence weight, descending:")
print(f"  {np.round(weights[:12], 4)} ...")
print(f"\n  {n_substantial} carry real valence character (> 0.01)")
print(f"  {n_orderable} are orderable at all (> 1e-6); beyond that the ranking is round-off")
print(f"\nN_POOL = {N_POOL} -> {'OK' if N_POOL <= n_orderable else 'OVERRUNS the valence space'}"
      f", cut at w[{N_POOL - 1}] = {weights[N_POOL - 1]:.4f}", end="")
if N_POOL < len(weights):
    gap = weights[N_POOL - 1] - weights[N_POOL]
    print(f", next {weights[N_POOL]:.4f}, gap {gap:.4f}")
else:
    print("  (whole virtual space, no cut)")

38 environment-free virtuals; fragment valence weight, descending:
  [0.9356 0.9252 0.2171 0.0819 0.026  0.0058 0.005  0.001  0.     0.     0.     0.    ] ...

  5 carry real valence character (> 0.01)
  8 are orderable at all (> 1e-6); beyond that the ranking is round-off

N_POOL = 5 -> OK, cut at w[4] = 0.0260, next 0.0058, gap 0.0202


## 5. Choose the CAS, freezing part of the fragment core

`select_cas_by_mp2_no` runs MP2 inside [fragment occupied + pool] and ranks **both** blocks by how
far their natural occupations have moved off the integers:

- **virtuals**, most occupied first — those are where correlation actually goes;
- **occupied**, most *depleted* first — with `N_CAS_OCC` set, the leading ones stay active and the
  rest become frozen CASCI core.

Ranking the occupied block by MP2 depletion rather than by orbital energy is the part that matters.
The highest-energy fragment occupied orbital is not necessarily the correlated one: a lone pair can
sit above a $\pi$ bond while contributing far less correlation, and freezing by energy would then
freeze the wrong orbital.

The printed occupations are how to judge the cut. The active ones should be visibly further from 2.0
than the frozen ones; if the frozen orbitals sit at 1.999 they were costing nothing, and if they are
as depleted as the active ones then `N_CAS_OCC` is too small.

In [8]:
sel = select_cas_by_mp2_no(mf, FRAGMENT, n_cas_vir=N_CAS_VIR, n_cas_occ=N_CAS_OCC,
                           n_pool=N_POOL, env_cols=env_cols, ovlp=ovlp)

occ_cols = np.where(mf.mo_occ > 0)[0]
frozen_core = np.setdiff1d(occ_cols, sel.cas_occ_cols)

print(f"all {len(occ_cols)} fragment occupied orbitals, ranked by MP2 depletion:")
print(f"  natural occupations {np.round(sel.fragment_natural_occ, 5)}")
print(f"  -> active in the CAS  : columns {sel.cas_occ_cols}, occupations "
      f"{np.round(sel.fragment_natural_occ[:N_ACT_OCC], 5)}")
if N_FROZEN_CORE:
    print(f"  -> frozen HF core     : columns {frozen_core}, occupations "
          f"{np.round(sel.fragment_natural_occ[N_ACT_OCC:], 5)}")
else:
    print("  -> frozen HF core     : none, every fragment occupied orbital stays active")
print(f"\nCAS virtuals              : {sel.cas_vir_cols}   MP2 natural occ "
      f"{np.round(sel.cas_natural_occ, 5)}")
print(f"\nCAS({sel.nelecas[0] + sel.nelecas[1]}e, {sel.ncas}o), columns {sel.mo_cas_idxs}")
assert not np.intersect1d(sel.mo_cas_idxs, env_cols).size, "CAS contains an environment orbital"

print(f"\n{report_cas_orbitals(mf.mol, sel.c_full[:, sel.mo_cas_idxs], FRAGMENT, N_ACT_OCC, mo_energy=mf.mo_energy[sel.mo_cas_idxs], occ_no=sel.cas_natural_occ)}")

all 4 fragment occupied orbitals, ranked by MP2 depletion:
  natural occupations [1.96098 1.98288 1.98921 1.99663]
  -> active in the CAS  : columns [0 1], occupations [1.96098 1.98288]
  -> frozen HF core     : columns [2 3], occupations [1.98921 1.99663]

CAS virtuals              : [4 5 6]   MP2 natural occ [0.03979 0.01719 0.01034]

CAS(4e, 5o), columns [0 1 4 5 6]

    idx  occ    energy  fragpop  valence  spread   MP2 nat occ   composition
      0  2.0   -1.3366    0.995    0.983    1.83                C0:0.32 O1:0.67
      1  2.0   -0.6571    0.987    0.959    1.35                C0:0.34 O1:0.64
      2  0.0    0.1353    0.957    0.925    2.11      0.03979   C0:0.61 O1:0.35
      3  0.0    0.1789    0.974    0.842    1.63      0.01719   C0:0.47 O1:0.50
      4  0.0    0.2156    0.579    0.213    2.38      0.01034   C0:0.42 O1:0.16
    fragment population: occupied min 0.987, virtual min 0.579 mean 0.837; virtual valence weight 1.98


## 6. CASCI, and what the active space is worth

`run_casci` selects the active orbitals by **column index** rather than letting `mcscf.CASCI` take a
contiguous window around the frontier, which after Huzinaga embedding can silently swallow an
environment orbital. `ncore` is where the frozen fragment orbitals of section 5 end up.

`diagnose_cas` then answers whether the CAS was worth choosing. $N_u$, the number of effectively
unpaired electrons, is the headline: ~0 means a single determinant and a wasted CAS, ~2 a
diradical. Acetaldehyde at equilibrium is a mildly correlated closed-shell molecule, so expect a
small but non-zero value — enough that the $\pi/\pi^*$ pair is doing something, far from a
diradical.

In [9]:
casci = run_casci(mf, sel.ncas, sel.nelecas, mo_coeff=sel.c_full, mo_cas_idxs=sel.mo_cas_idxs)
e_casci_in_dft = casci.e_tot + env_plus_corr

assert casci.ncore == N_FROZEN_CORE, f"expected {N_FROZEN_CORE} core orbitals, got {casci.ncore}"
print(f"CASCI ncore = {casci.ncore} (frozen fragment orbitals), ncas = {casci.ncas}\n")
print(f"CASCI-in-DFT  = {e_casci_in_dft:.10f}")
print(f"HF-in-DFT     = {e_hf_in_dft:.10f}   "
      f"(correlation recovered {e_casci_in_dft - e_hf_in_dft:+.6f} Ha)")

print(f"\n{diagnose_cas(casci, verbose=False).report()}")
print(f"\nactive space vs the frozen environment: "
      f"{environment_overlap(sel.c_full[:, sel.mo_cas_idxs], emb.C_full_reidx[:, emb.env_idx_occ], ovlp):.2e}")

CASCI ncore = 2 (frozen fragment orbitals), ncas = 5

CASCI-in-DFT  = -153.4966005906
HF-in-DFT     = -153.4345868701   (correlation recovered -0.062014 Ha)

  CASCI energy of the subsystem : 46.10744573 Ha
  natural occupations           : [1.9813 1.9319 0.0679 0.0188 0.0002]
  effectively unpaired electrons: 0.1737   (nonlinear 0.0373)
  entropy of the occupations    : 0.4039 nats
  aufbau determinant weight     : 0.9547   so 1 - c0^2 = 0.0453
  largest determinant weight    : 0.9547
  summed orbital entropy        : 0.6003 nats
  per-orbital entropy           : [0.2072 0.0926 0.2068 0.0928 0.001 ]
  verdict                       : moderately correlated

active space vs the frozen environment: 5.00e-16


## 7. The fragment as a qubit Hamiltonian

`get_mo_integrals` returns the CAS integrals in the embedded MO basis with $v_\text{emb}$ inside the
one-electron part, and folds *everything* outside the CAS into `ecore` — the frozen fragment core of
section 5 included. So the constant to carry is

$$E_\text{shift} = \texttt{env\_plus\_corrections} + \texttt{ecore},$$

and exact diagonalisation must reproduce the CASCI energy. This is the check that the frozen-core
bookkeeping is right: if `ecore` had missed the frozen orbitals' contribution, the agreement below
would fail by hartrees rather than $10^{-13}$.

The Pauli Hamiltonian does not conserve particle number on its own, so penalties
$(\hat N_\alpha - n_\alpha)^2 + (\hat N_\beta - n_\beta)^2$ pin the ground state to the right sector.

In [10]:
from nbed.hamiltonian import (build_molecular_H, build_number_operator,
                              build_spin_integrals)
from openfermion import get_sparse_operator
from scipy.sparse.linalg import eigsh

ecore, h1e, eri = emb.get_mo_integrals(mf, sel.c_full, sel.ncas, sel.nelecas,
                                       mo_cas_idxs=sel.mo_cas_idxs)
h1e_spin, eri_spin = build_spin_integrals(h1e, eri, sel.ncas)
nqubits = 2 * sel.ncas
assert nqubits <= 14, f"{nqubits} qubits is too many for dense diagonalisation"

Hq = build_molecular_H(env_plus_corr + ecore, h1e_spin, eri_spin)
Na, Nb = build_number_operator(nqubits, type="qubit_jw")
H_sym = Hq + (Na - sel.nelecas[0]) ** 2 + (Nb - sel.nelecas[1]) ** 2
eigvals = eigsh(get_sparse_operator(H_sym).real, k=2, which="SA")[0]

print(f"{nqubits} qubits, {len(Hq.terms)} Pauli terms")
print(f"qubit ground state = {eigvals[0]:.10f}")
print(f"CASCI-in-DFT       = {e_casci_in_dft:.10f}")
print(f"difference         = {eigvals[0] - e_casci_in_dft:+.2e}")
assert abs(eigvals[0] - e_casci_in_dft) < 1e-9

10 qubits, 444 Pauli terms
qubit ground state = -153.4966005906
CASCI-in-DFT       = -153.4966005906
difference         = +5.68e-14


## Summary

The whole path, in one configurable pass: global DFT on acetaldehyde, the carbonyl group embedded
with 8 of its 24 electrons, a CAS chosen inside the fragment by MP2 natural occupation with part of
the fragment core frozen, and the result reproduced as a 10-qubit Hamiltonian.

What each check was for:

1. **DFT-in-DFT reproduces global DFT.** The partition and projector are exact, not approximate.
   With `"huz"` this is $\sim 10^{-11}$ Ha; with `"mu"` the residual is the $\mathcal{O}(1/\mu)$
   error, which is why Huzinaga is the default recommendation.
2. **The pool sits inside the fragment's valence space.** Overrunning it makes the trailing pool
   orbitals arbitrary and the final energy irreproducible at $10^{-4}$ Ha, which would otherwise
   look like the projector being inexact when it is not.
3. **The CAS contains no environment orbital**, checked by column intersection and by
   `environment_overlap` against the originally frozen environment.
4. **Exact diagonalisation reproduces CASCI to $\sim 10^{-13}$ Ha** with `ncore = 2`, which is what
   confirms the frozen fragment core is correctly folded into `ecore`. It stays below $10^{-10}$ Ha
   for every setting of `N_CAS_OCC`, including `None`.

Things to try by editing the first cell:

- `PROJ_TYPE`: `"huz"` and `"huz+shift"` agree to $10^{-10}$ Ha; `"mu"` differs by
  $1.2\times10^{-7}$. Compare the `env_cols` line between them — scattered versus contiguous.
- `N_CAS_OCC = None` makes every fragment occupied orbital active, so `ncore = 0` and the qubit count
  rises to 14. It recovers ~19 mHa more correlation than the frozen-core run, which is the price of
  freezing; `N_CAS_OCC = 1` gives away ~32 mHa in the other direction.
- `N_CAS_OCC = 1` freezes three of the four. The frozen orbitals are chosen by MP2 depletion, so
  check what the ordering does before assuming the highest-energy one stays active.
- `N_POOL = 9` trips the valence-space warning from section 4, and moves the answer by ~1 mHa for no
  good reason. Growing the pool only helps once the fragment is big enough to have the valence
  virtuals to fill it, which `N_OCC_FRAG` controls.